# evaluate

> Score any backend that answers in the Jev schema: the typed-decisions test split, the zero-shot suites, and behaviour probes from failures seen in the wild.

In [ ]:
#| default_exp evaluate

In [ ]:
#| export
import json, time
from collections import defaultdict

import numpy as np

from sysone.core import *
from sysone.losses import answer_metrics, ece_score, auroc
from sysone.metrics import Preds, as_metric

In [ ]:
#| hide
from fastcore.test import test_eq, test_close, test_fail
from pathlib import Path
import logging, torch, sysone
logging.getLogger("transformers").setLevel(logging.ERROR)
import datasets, transformers; datasets.disable_progress_bars(); transformers.utils.logging.disable_progress_bar()

Evaluation here only needs `predict(state, questions)` returning answers in the Jev schema, so the same functions score a sysone `Decider`, a `GlinerDecider`, a `laya.Agent` (whose result wraps the answers in `{"answers": ...}`) or a client of any `/v1/systemone` endpoint, with the definitions of Laya's evaluation cell. A small trained model to score:

In [ ]:
from sysone.data import TypedDecisions
from sysone.learner import decision_learner

In [ ]:
recs = [json.loads(l) for l in Path("fixtures/typed_decisions_tiny.jsonl").read_text().splitlines()]
data = TypedDecisions.from_records(recs[:15], calib=0)
torch.manual_seed(0)
learn = decision_learner(data, "fixtures/tiny-modernbert", train="full", preset="cpu", loss="soft_ce")
learn.fit(3, lr=2e-3)
jev = learn.decider()

{'loss': '1.211', 'grad_norm': '0.3333', 'learning_rate': '0.002', 'epoch': '1'}
{'loss': '1.233', 'grad_norm': '0.8835', 'learning_rate': '0.002', 'epoch': '2'}
{'loss': '1.245', 'grad_norm': '0.5154', 'learning_rate': '0.002', 'epoch': '3'}
{'train_runtime': '1.443', 'train_samples_per_second': '155.9', 'train_steps_per_second': '20.79', 'train_loss': '1.23', 'epoch': '3'}


100%|##########| 30/30 [00:01<00:00, 20.79it/s]


## evaluate

In [ ]:
#| export
def answers_of(result) -> dict:
    "The answers dict of a predict result (Laya's runtime wraps it in {'model', 'answers', 'usage'})"
    if isinstance(result, dict) and "answers" in result and isinstance(result["answers"], dict) and "usage" in result:
        return result["answers"]
    return result

def evaluate(model, records, by:str|None=None, batch_size:int|None=None, metrics:list|None=None) -> dict:
    "The notebook's metrics, latency, any more `metrics` (functions of a `Preds`) and, with `by`, accuracy per group, for `model` over records with gold"
    records = [load_record(r) for r in records]
    lat, triples = [], []
    if batch_size and hasattr(model, "predict_batch") and len({json.dumps(r["questions"], sort_keys=True) for r in records}) == 1:
        for i in range(0, len(records), batch_size):
            chunk = records[i:i + batch_size]
            t0 = time.perf_counter(); res = model.predict_batch([r["state"] for r in chunk], chunk[0]["questions"])
            lat += [(time.perf_counter() - t0) * 1000 / len(chunk)] * len(chunk)
            triples += [(r["questions"], answers_of(a), r.get("gold") or {}) for r, a in zip(chunk, res)]
    else:
        for r in records:
            t0 = time.perf_counter(); a = answers_of(model.predict(r["state"], r["questions"]))
            lat.append((time.perf_counter() - t0) * 1000)
            triples.append((r["questions"], a, r.get("gold") or {}))
    m = answer_metrics(triples) | {"latency_p50_ms": float(np.percentile(lat, 50)), "latency_p95_ms": float(np.percentile(lat, 95))}
    if by:
        groups = defaultdict(list)
        for r, t in zip(records, triples): groups[r.get(by)].append(t)
        m[f"accuracy_by_{by}"] = {g: round(answer_metrics(ts)["accuracy"], 4) for g, ts in sorted(groups.items(), key=lambda kv: str(kv[0]))}
    if metrics:
        p = Preds.from_answers(records, [a for _, a, _ in triples])
        m |= {mm.name: mm(p) for mm in map(as_metric, metrics)}
    return m

In [ ]:
res = evaluate(jev, recs[15:], by="workflow")
res

{'accuracy': 0.08,
 'soft_accuracy': 0.4155073242587161,
 'brier': 0.13878789392453206,
 'kl': 0.20540641413598718,
 'tv': 0.2551017098356831,
 'ece': 0.26305675984764754,
 'score_mae': 0.6231583768616458,
 'within_one': 0.9,
 'n': 25,
 'accuracy_choice': 0.0,
 'accuracy_score': 0.0,
 'accuracy_noul': 0.2,
 'macro_f1': 0.05714285714285714,
 'latency_p50_ms': 8.903915993869305,
 'latency_p95_ms': 9.540083399042487,
 'accuracy_by_workflow': {'security_incidents': 0.08}}

In [ ]:
test_eq(res["n"], 25)
assert 0 <= res["accuracy"] <= 1 and res["latency_p50_ms"] > 0
test_eq(set(res["accuracy_by_workflow"]) <= {r["workflow"] for r in recs}, True)
test_eq(evaluate(jev, recs[15:], batch_size=2)["accuracy"], res["accuracy"])   # one workflow, one question set: batched

Any metric of the [metrics](04a_metrics.ipynb) notebook works here too, from the answers alone: `metrics=` adds a column for each, computed on `Preds.from_answers`:

In [ ]:
from sysone.metrics import F1Score, CohenKappa, Metric, noul_auroc

In [ ]:
more = evaluate(jev, recs[15:], metrics=[F1Score("weighted"), Metric(CohenKappa("quadratic"), qtype="score"), noul_auroc])
test_eq(more["accuracy"], res["accuracy"])
{k: round(more[k], 3) for k in ("accuracy", "macro_f1", "f1_score_weighted", "cohen_kappa_quadratic_score", "noul_auroc")}

{'accuracy': 0.08,
 'macro_f1': 0.057,
 'f1_score_weighted': 0.046,
 'cohen_kappa_quadratic_score': 0.0,
 'noul_auroc': 0.312}

### typed-decisions

`typed_decisions_report` scores the test split, 400 cases and 2,000 decisions, per workflow, next to the reference rows of the dataset card, all measured on the same split. Read 0.52 as the floor and 0.735, the teacher's self-agreement, as saturation; the card's advice is to read KL and Brier as well, since a prior that ignores the input has the best ECE.

In [ ]:
#| export
TYPED_DECISIONS_REFERENCE = [
    {"model": "Prior (ignores the input)", "kind": "reference", "accuracy": 0.470, "brier": 0.189, "ece": 0.088},
    {"model": "ModernBERT-base (149M)", "kind": "specialist", "accuracy": 0.646, "brier": 0.119, "ece": 0.179},
    {"model": "TypeSafe Jev 1.13.0", "kind": "general, zero-shot", "accuracy": 0.727, "brier": 0.148, "ece": 0.144},
    {"model": "Teacher self-agreement", "kind": "ceiling", "accuracy": 0.735},
    {"model": "Laya fine-tuned (2×T4 notebook)", "kind": "fine-tuned", "accuracy": 0.766},
]

def typed_decisions_report(model, split:str="test", n:int|None=None, name:str="this model") -> dict:
    "Metrics on typed-decisions `split`, per workflow, beside the dataset card's reference rows"
    from datasets import load_dataset
    ds = load_dataset("LocalLLaMA/typed-decisions", "all", split=split)
    if n is not None: ds = ds.select(range(min(n, len(ds))))
    m = evaluate(model, list(ds), by="workflow")
    return {"metrics": m, "table": TYPED_DECISIONS_REFERENCE + [{"model": name, "kind": "sysone", **{k: round(m[k], 3) for k in ("accuracy", "brier", "ece")}}]}

In [ ]:
#| eval: false
report = typed_decisions_report(jev)

### Zero-shot suites

`zero_shot` scores a model on the classification suites of `sysone.datasets`, with no task-specific training, the setting Laya and Jev publish figures for:

In [ ]:
#| export
def zero_shot(model, suites=("ag_news", "emotion", "banking77", "massive", "xnli"), n:int=500, seed:int=0) -> dict:
    "Accuracy and calibration on each zero-shot suite (a seeded sample of `n` test rows)"
    from sysone.datasets import suite_records
    out = {}
    for s in suites:
        m = evaluate(model, suite_records(s, n=n, seed=seed))
        out[s] = {k: round(m[k], 4) for k in ("accuracy", "ece", "brier", "latency_p50_ms")}
    return out

In [ ]:
#| eval: false
zero_shot(jev, n=200)

## Behaviour probes

[semantic-browser](https://github.com/koriym/semantic-browser) drove a hypermedia agent with Laya zero-shot and documented four failures, with deterministic repeats. `behaviors` turns them into regression probes; each returns whether it passed and what it saw, and a model should be expected to fail some of them until trained for it:

- **negation**: an option whose description says it *cannot* serve the goal ("cannot be used to find the reviewers of a specific manuscript") won with 0.79 for a goal it excludes; the correct option must win, and adding the negation to an option must lower its probability;
- **uri**: a URI in the state (`"At /index. Goal: list all books."`) flipped the winner to the `index` link, 0.912, which lost to `books` without it; the argmax must be `books` both ways;
- **candidate set**: the winner between two options changed when a third, irrelevant one was removed (books 0.627 with `index` present, authors 0.601 without); the order of the two must not flip;
- **confidence**: zero-shot confidence did not track correctness (a wrong answer at 0.96, a right one at 0.29); on records with gold, the probe reports the AUROC of `answer_confidence` against correctness, which should clear 0.5 by a margin.

The first probe's descriptions are English renderings of the Japanese ones in the semantic-browser trace; the others use its English strings.

In [ ]:
#| export
_LINK = "Which link should I follow next?"

def _probs(model, state, criteria, qid="rel"):
    a = answers_of(model.predict(state, {qid: {"type": "choice", "instructions": _LINK, "criteria": criteria}}))[qid]
    return a["probabilities"], a["choice"]

def probe_negation(model) -> dict:
    "The option that negates the goal must not win, and adding a negation must lower an option's probability"
    state = " Goal: Find the name of the referee who argued against publishing the paper titled 'Affordance Density in Machine-Readable Hypermedia'."
    crit = {"goManuscriptList": "Manuscripts under consideration; these are not yet published.",
            "goIssueList": "Published issues; published papers are reached from here.",
            "goReviewerPool": "The reviewer roster; it cannot be used to find the reviewers of a specific manuscript."}
    p, choice = _probs(model, state, crit)
    plain = dict(crit, goReviewerPool="The reviewer roster.")
    p0, _ = _probs(model, state, plain)
    passed = choice == "goIssueList" and p["goReviewerPool"] < p0["goReviewerPool"]
    return {"probe": "negation", "passed": bool(passed), "choice": choice, "probabilities": p,
            "p_roster_without_negation": p0["goReviewerPool"]}

_API = {"books": "The complete collection of books in the library.", "authors": "Every author who has written at least one book.",
        "index": "The entry page of the API."}

def probe_uri(model) -> dict:
    "A URI in the state must not flip the winner"
    with_uri, c1 = _probs(model, "At /index. Goal: list all books.", _API)
    without, c2 = _probs(model, "Goal: list all books.", _API)
    return {"probe": "uri", "passed": c1 == c2 == "books", "choice_with_uri": c1, "choice_without": c2,
            "p_index_with_uri": with_uri["index"]}

def probe_candidate_set(model) -> dict:
    "Removing an irrelevant option must not flip the order of the other two"
    state = "Goal: find who wrote Norwegian Wood."
    p3, _ = _probs(model, state, _API)
    p2, _ = _probs(model, state, {k: _API[k] for k in ("books", "authors")})
    flip = (p3["books"] > p3["authors"]) != (p2["books"] > p2["authors"])
    return {"probe": "candidate_set", "passed": not flip, "with_index": p3, "without_index": p2}

def probe_confidence(model, records, margin:float=0.05) -> dict:
    "Confidence must track correctness: AUROC of answer_confidence against being right"
    conf, right = [], []
    for r in map(load_record, records):
        ans = answers_of(model.predict(r["state"], r["questions"]))
        for qid, qd in r["questions"].items():
            if qid not in (r.get("gold") or {}): continue
            q = Question.from_dict(qd, name=qid)
            t, label = q.target(r["gold"][qid])
            a = ans[qid]
            pred = answer_index(q, answer_probs(q, a))
            conf.append(a["answer_confidence"]); right.append(pred == label)
    au = auroc(conf, right)
    return {"probe": "confidence", "passed": bool(au > 0.5 + margin), "auroc": au, "ece": ece_score(conf, right), "n": len(conf)}

def behaviors(model, records=None) -> list:
    "Run every behaviour probe (the confidence probe needs records with gold)"
    out = [probe_negation(model), probe_uri(model), probe_candidate_set(model)]
    if records is not None: out.append(probe_confidence(model, records))
    return out

In [ ]:
test_close(auroc([0.1, 0.4, 0.35, 0.8], [0, 0, 1, 1]), 0.75)
test_eq(auroc([0.5, 0.5], [0, 1]), 0.5)
probes = behaviors(jev, recs[15:])
[(p["probe"], p["passed"]) for p in probes]

[('negation', False),
 ('uri', True),
 ('candidate_set', True),
 ('confidence', True)]

In [ ]:
test_eq([p["probe"] for p in probes], ["negation", "uri", "candidate_set", "confidence"])
test_eq(all(isinstance(p["passed"], bool) for p in probes), True)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()